# 03 - Regressão: Modelagem Preditiva

## Objetivo
Este notebook aplica um modelo de regressão linear para prever valores de exportação (VL_FOB) baseado em diversas variáveis preditoras.

### Etapas do Processo:
1. Preparação dos dados para modelagem
2. Divisão entre treino e teste
3. Aplicação de regressão linear
4. Avaliação do modelo (MSE, MAE, R², RMSE)
5. Visualização dos resultados

### Pré-requisitos
Execute primeiro os notebooks:
- **01_Extract.ipynb**
- **02_Transform.ipynb**

## Importações e Configurações

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import os

# Configurações
OUTPUT_DIR = '../data/output'
DADOS_TRANSFORMADOS = os.path.join(OUTPUT_DIR, 'dados_transformados.csv')

# Configurar matplotlib
%matplotlib inline

print("✓ Bibliotecas importadas com sucesso!")

## Carregar Dados Transformados

In [ ]:
if os.path.exists(DADOS_TRANSFORMADOS):
    df = pd.read_csv(DADOS_TRANSFORMADOS)
    print(f"✓ Dados carregados: {len(df):,} registros")
    print(f"  Colunas: {len(df.columns)}")
else:
    raise FileNotFoundError(f"Execute primeiro o notebook 02_Transform.ipynb!")

## Preparação dos Dados para Modelagem

In [ ]:
print("=== PREPARAÇÃO DOS DADOS PARA MODELAGEM ===")

# Verificar se a coluna id_product existe
if 'id_product' in df.columns:
    produtos = df['id_product'].unique()
    print(f"Produtos encontrados: {len(produtos)} tipos diferentes")
    print(f"Exemplos: {list(produtos[:5])}")
    produto_alvo = "TODOS_OS_PRODUTOS"
    df_analise = df.copy()
else:
    print("Coluna id_product não encontrada, processando todos os dados...")
    produto_alvo = "TODOS_OS_DADOS"
    df_analise = df.copy()

print(f"\nRegistros para análise: {len(df_analise):,}")

## Seleção de Variáveis

In [ ]:
# Variáveis preditoras (features)
features = ['CO_ANO', 'CO_MES', 'CO_NCM', 'CO_UNID', 'CO_PAIS', 'SG_UF_NCM', 'CO_VIA', 'CO_URF', 'QT_ESTAT', 'KG_LIQUIDO']

# Verificar se todas as features existem
features_disponiveis = [f for f in features if f in df_analise.columns]
print(f"Features disponíveis: {len(features_disponiveis)}/{len(features)}")

if len(features_disponiveis) < len(features):
    print(f"Features faltando: {set(features) - set(features_disponiveis)}")
    print("Usando apenas features disponíveis...")

# Variável alvo (target)
target = 'VL_FOB'

if target not in df_analise.columns:
    raise ValueError(f"Coluna alvo {target} não encontrada!")

X = df_analise[features_disponiveis]
y = df_analise[target]

print(f"\n✓ Variáveis selecionadas:")
print(f"  Features: {features_disponiveis}")
print(f"  Target: {target}")

## Divisão Treino/Teste

In [ ]:
print("=== DIVISÃO TREINO/TESTE ===")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✓ Divisão concluída:")
print(f"  Treino: {len(X_train):,} registros ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Teste: {len(X_test):,} registros ({len(X_test)/len(X)*100:.1f}%)")

## Criação e Treinamento do Modelo

In [ ]:
print("=== CRIAÇÃO E TREINAMENTO DO MODELO ===")

# Identificar colunas categóricas
categorical_cols = ['CO_PAIS', 'SG_UF_NCM']
categorical_cols = [col for col in categorical_cols if col in X.columns]

if categorical_cols:
    print(f"Colunas categóricas: {categorical_cols}")
    
    # Definir pipeline com pré-processamento
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ],
        remainder='passthrough'
    )
    
    modelo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ])
else:
    print("Nenhuma coluna categórica encontrada, usando modelo simples...")
    modelo = LinearRegression()

# Treinar modelo
print("\nTreinando modelo...")
modelo.fit(X_train, y_train)
print("✓ Modelo treinado com sucesso!")

## Avaliação do Modelo

In [ ]:
print("=== AVALIAÇÃO DO MODELO ===")

# Fazer previsões
y_pred = modelo.predict(X_test)

# Calcular métricas
mse = mean_squared_error(y_test, y_pred)
mae = abs(y_test - y_pred).mean()
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"✓ Métricas de avaliação:")
print(f"  MSE (Erro Quadrático Médio): {mse:,.2f}")
print(f"  MAE (Erro Absoluto Médio): R$ {mae:,.2f}")
print(f"  RMSE (Raiz do EQM): R$ {rmse:,.2f}")
print(f"  R² (Coeficiente de Determinação): {r2:.4f}")

# Exibir primeiras previsões
print(f"\n✓ Primeiras 5 previsões vs valores reais:")
for i in range(5):
    print(f"  Real: R$ {y_test.iloc[i]:,.2f} | Previsto: R$ {y_pred[i]:,.2f}")

## Visualização dos Resultados

In [ ]:
print("=== CRIANDO VISUALIZAÇÕES ===")

# Calcular estatísticas para visualização
total_ncm = df['CO_NCM'].nunique() if 'CO_NCM' in df.columns else 0
total_paises = df['CO_PAIS'].nunique() if 'CO_PAIS' in df.columns else 0
total_estados = df['SG_UF_NCM'].nunique() if 'SG_UF_NCM' in df.columns else 0
total_urfs = df['CO_URF'].nunique() if 'CO_URF' in df.columns else 0
valor_total_fob = df['VL_FOB'].sum()
peso_total_kg = df['KG_LIQUIDO'].sum() if 'KG_LIQUIDO' in df.columns else 0

# Calcular top produtos e países se existirem as colunas
if 'id_product' in df.columns:
    top_produtos = df.groupby('id_product')['VL_FOB'].sum().sort_values(ascending=False).head(10)
else:
    top_produtos = df.groupby('CO_NCM')['VL_FOB'].sum().sort_values(ascending=False).head(10)

if 'id_country' in df.columns:
    top_paises = df.groupby('id_country')['VL_FOB'].sum().sort_values(ascending=False).head(10)
elif 'CO_PAIS' in df.columns:
    top_paises = df.groupby('CO_PAIS')['VL_FOB'].sum().sort_values(ascending=False).head(10)
else:
    top_paises = pd.Series()

# Função para truncar nomes longos
def truncar_nome(nome, max_len=35):
    if len(str(nome)) > max_len:
        return str(nome)[:max_len-3] + '...'
    return str(nome)

# Truncar nomes
top_produtos.index = [truncar_nome(x) for x in top_produtos.index]
top_paises.index = [truncar_nome(x, max_len=25) for x in top_paises.index]

print("✓ Estatísticas calculadas!")

In [ ]:
# Criar gráfico completo
plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#f8f9fa')

# Título principal
fig.suptitle(f' Análise de Regressão - COMEX Stat 2025\n{produto_alvo}', 
             fontsize=16, fontweight='bold', color='#2c3e50', y=0.98)

# === SUBPLOT 1: Estatísticas do Dataset ===
ax1 = plt.subplot(2, 2, 1)
ax1.axis('off')
ax1.set_facecolor('#f8f9fa')

dataset_stats_text = f'''

                  ESTATÍSTICAS DO DATASET                   

                                                                
       Total de Registros:      {len(df):>15,}                  
       Produtos NCM Únicos:     {total_ncm:>15,}                  
       Países de Destino:       {total_paises:>15,}                  
       Estados (UF):            {total_estados:>15,}                  
       URFs Utilizadas:         {total_urfs:>15,}                  
                                                                

                     TOTAIS DE EXPORTAÇÃO                  

                                                                
       Valor Total FOB:    R$ {valor_total_fob:>18,.2f}        
        Peso Total (kg):   {peso_total_kg:>18,.2f}               
                                                                

                     MÉDIAS POR REGISTRO                   

                                                                
       Média FOB:         R$ {valor_total_fob/len(df):>18,.2f}        
        Média Peso (kg):   {peso_total_kg/len(df):>18,.2f}               
                                                                

'''

ax1.text(0.5, 0.5, dataset_stats_text, transform=ax1.transAxes, fontsize=9,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='white', 
         edgecolor='#28a745', linewidth=2, alpha=0.95), linespacing=1.1)

# === SUBPLOT 2: Top 10 Produtos ===
ax2 = plt.subplot(2, 2, 2)
ax2.set_facecolor('#ffffff')

if len(top_produtos) > 0:
    top_produtos.plot(kind='barh', color='#3498db', ax=ax2, edgecolor='black', linewidth=0.5)
    ax2.set_xlabel('Valor FOB (R$)', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Produto', fontsize=10, fontweight='bold')
    ax2.set_title(' Top 10 Produtos por Valor FOB', fontsize=12, fontweight='bold', pad=10)
    ax2.tick_params(axis='y', labelsize=8)
    ax2.invert_yaxis()
    ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R${x/1e6:.0f}M'))
else:
    ax2.text(0.5, 0.5, 'Dados não disponíveis', ha='center', va='center')

# === SUBPLOT 3: Top 10 Países ===
ax3 = plt.subplot(2, 2, 3)
ax3.set_facecolor('#ffffff')

if len(top_paises) > 0:
    top_paises.plot(kind='barh', color='#e74c3c', ax=ax3, edgecolor='black', linewidth=0.5)
    ax3.set_xlabel('Valor FOB (R$)', fontsize=10, fontweight='bold')
    ax3.set_ylabel('País de Destino', fontsize=10, fontweight='bold')
    ax3.set_title(' Top 10 Países por Valor FOB', fontsize=12, fontweight='bold', pad=10)
    ax3.tick_params(axis='y', labelsize=8)
    ax3.invert_yaxis()
    ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R${x/1e6:.0f}M'))
else:
    ax3.text(0.5, 0.5, 'Dados não disponíveis', ha='center', va='center')

# === SUBPLOT 4: Estatísticas do Modelo ===
ax4 = plt.subplot(2, 2, 4)
ax4.axis('off')
ax4.set_facecolor('#f8f9fa')

model_stats_text = f'''

                     ESTATÍSTICAS DO MODELO                 

                                                                  
       Registros Analisados:    {len(df_analise):>15,}          
       Total no Dataset:        {len(df):>15,}                  
                                                                  

                     MÉTRICAS DE ERRO                       

                                                                  
       MAE (Erro Absoluto):     R$ {mae:>15,.2f}                
       RMSE:                    R$ {rmse:>15,.2f}                
       MSE:                     R$ {mse:>15,.2f}                
                                                                  

                     MÉTRICAS DE QUALIDADE                  

                                                                  
       R² Score:                {r2:>15.4f}                
       Acurácia do Modelo:      {r2*100:>14.1f}%                
                                                                  

'''

ax4.text(0.5, 0.5, model_stats_text, transform=ax4.transAxes, fontsize=9,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='white', 
         edgecolor='#3498db', linewidth=2, alpha=0.95), linespacing=1.1)

# Ajustar layout
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Salvar gráfico
grafico_path = os.path.join(OUTPUT_DIR, 'grafico_previsao_completo.png')
plt.savefig(grafico_path, dpi=300, bbox_inches='tight')
print(f"✓ Gráfico salvo em: {grafico_path}")

plt.show()

## Salvar Dados e Resultados

In [ ]:
# Adicionar coluna de previsão ao dataframe
df_com_previsao = df_analise.copy()
df_com_previsao['VL_FOB_PREVISTO'] = modelo.predict(X)

# Salvar dados com previsões
output_file = os.path.join(OUTPUT_DIR, 'dados_com_previsoes.csv')
df_com_previsao.to_csv(output_file, index=False)

print(f"✓ Dados com previsões salvos em: {output_file}")
print(f"  Total de registros: {len(df_com_previsao):,}")
print(f"  Colunas: {len(df_com_previsao.columns)}")